<a href="https://colab.research.google.com/github/bvsvntv/ai-ml-learning/blob/main/notebooks/lang-graph/ltm_in_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Using existing memories

In [137]:
!pip install -U langchain-openai

In [138]:
import uuid
from typing import List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from google.colab import userdata

In [139]:
api_key = userdata.get("OPENAI_API_KEY")
model = ChatOpenAI(api_key=api_key, model="gpt-4o-mini")

In [140]:
# LTM store
store = InMemoryStore()

namespace = ("user", "u1", "details")

# seed memories before running graph
james_bond = {
    "profile_1": {"data": "Name: James Bond"},
    "profile_2": {"data": "Profession: MI6 Secret Agent"},
    "profile_3": {"data": "Code Name: 007"},
    "preference_1": {"data": "Prefers Martinis, shaken not stirred"},
    "preference_2": {"data": "Likes Aston Martin cars"},
    "preference_3": {"data": "Enjoys high-tech gadgets"},
    "project_1": {"data": "Mission: Protect British national security"},
    "project_2": {"data": "Specialization: Espionage and covert operations"}
}

for key, value in james_bond.items():
  store.put(namespace, key, value)

In [141]:
memories = store.search(namespace)
for memory in memories:
  print(memory.value)

{'data': 'Name: James Bond'}
{'data': 'Profession: MI6 Secret Agent'}
{'data': 'Code Name: 007'}
{'data': 'Prefers Martinis, shaken not stirred'}
{'data': 'Likes Aston Martin cars'}
{'data': 'Enjoys high-tech gadgets'}
{'data': 'Mission: Protect British national security'}
{'data': 'Specialization: Espionage and covert operations'}


In [142]:
# system prompt
SYSTEM_PROMPT = """You are a helpful assistant with memory capabilities.
If user-specific memory is available, use it to personalize your responses based
on what you know about the user.

Your goal is to provide relevant, friendly, and tailored assistance that reflects
the user's preferences, context, and past interactions.

If the user's name or relevant personal context is available, always personalize
your responses by:
  - Always address the user by their name (e.g., "Sure, James...") when appropriate
  - Adjusting the tone to feel friendly, natural, and directly aimed at the user

Avoid generic phrasing when personalization is possible.

Use personalization especially in:
  - Greetings and transitions
  - Help or tailored guidance
  - Follow-up messages that continue from the past context

Always ensure that personalization is based only on know user details and not assumed.

In the end suggest 3 relevant further questions based on the current response and user profile.

The user's memory (which may be empty) is provided as: {user_details}
"""

In [143]:
def chat_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
  user_id = config["configurable"]["user_id"]

  # fetch user details
  namespace = ("user", user_id, "details")
  memories = store.search(namespace)

  # convert memory items into string blob
  if memories:
    user_details = "\n".join(f"- {memory.value.get("data", "")}" for memory in memories)
  else:
    user_details = ""

  system_prompt = SYSTEM_PROMPT.format(user_details=user_details)

  system_msg = SystemMessage(content=system_prompt)

  response = model.invoke([system_msg] + state["messages"])
  return {"messages": [response]}

In [144]:
graph = StateGraph(MessagesState)
graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

workflow = graph.compile(store=store)

In [145]:
# invoke workflow
config = {"configurable": {"user_id": "u1"}}

result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Who am I and what are my preferences?"}]},
    config
)
print(result["messages"][-1].content)

You're James Bond, also known by your code name 007. You're a secret agent for MI6, specializing in espionage and covert operations to protect British national security. You prefer your martinis shaken, not stirred, and you're a fan of Aston Martin cars and high-tech gadgets. 

If there's anything specific you'd like to discuss or explore, just let me know! 

Here are some questions to consider:
1. Are you currently working on any missions that you’d like to share?
2. What’s your favorite Aston Martin model?
3. Which high-tech gadget do you find most essential in your operations?


### Creating new memories

In [146]:
# LTM store
store2 = InMemoryStore()

In [147]:
# LLM for extracting information that worth remembering
extractor_llm = ChatOpenAI(api_key=api_key, model="gpt-4o-mini", temperature=0)

In [148]:
class MemoryDecision(BaseModel):
  should_write: bool = Field(description="Whether to store any memories")
  memories: List[str] = Field(default_factory=list, description="Atomic user memories to store")

In [149]:
memory_extractor = extractor_llm.with_structured_output(MemoryDecision)

In [150]:
def remember_only_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
  user_id = config["configurable"]["user_id"]

  namespace = ("user", user_id, "details")

  # take latest user message
  last_msg = state["messages"][-1].content

  # Let LLM decide what to store
  decision: MemoryDecision = memory_extractor.invoke([
      SystemMessage(content=(
          "Extract LONG-TERM memories from the user's message.\n"
          "Only store stable, user-specific info (identity, preferences).\n"
          "Do NOT store transient info.\n"
          "Return should_write=false if nothing is worth storing.\n"
          "Each memory should be a short atomic sentence."
      )),
      {"role": "user", "content": last_msg}
  ])

  # write to LTM store
  if decision.should_write:
    for memory in decision.memories:
      store2.put(namespace, str(uuid.uuid4()), {"data": memory})

  # IMPORTANT: we are NOT using memory, not even responding with the LLM.
  # We just return a fixed acknowledgement.
  return {"messages": [{"role": "assistant", "content": "Noted."}]}

In [151]:
graph2 = StateGraph(MessagesState)

graph2.add_node("remember_only_node", remember_only_node)

graph2.add_edge(START, "remember_only_node")
graph2.add_edge("remember_only_node", END)

workflow2 = graph2.compile(store=store2)

In [152]:
config2 = {"configurable": {"user_id": "u1"}}

response = workflow2.invoke({"messages": [{"role": "user", "content": "Hi, My name is James Bond."}]}, config2)
print("Assistant:", response["messages"][-1].content)

Assistant: Noted.


In [153]:
response2 = workflow2.invoke({"messages": [{"role": "user", "content": "I'm MI6 Secret Agent."}]}, config2)
print("Assistant:", response2["messages"][-1].content)

Assistant: Noted.


In [154]:
response3 = workflow2.invoke({"messages": [{"role": "user", "content": "My code name is 007."}]}, config2)
print("Assistant:", response3["messages"][-1].content)

Assistant: Noted.


In [155]:
memories = store2.search(("user", "u1", "details"))
print("-"*36,"MEMORIES","-"*36)
for memory in memories:
  print(memory.value)

------------------------------------ MEMORIES ------------------------------------
{'data': "User's name is James Bond."}
{'data': "User's code name is 007."}


### Create new memories (without duplication)

In [156]:
store3 = InMemoryStore()

In [157]:
# model responsible for extracting candidate memories and detect duplication
memory_llm = ChatOpenAI(api_key=api_key, model="gpt-4o-mini", temperature=0)

In [158]:
class MemoryElement(BaseModel):
  text: str = Field(description="Atomic user memory in short sentence")
  is_new: bool = Field(description="True if this memory is NEW and should be stored. False if duplicate/already known.")

In [159]:
class MemoryDecision2(BaseModel):
  should_write: bool = Field(description="Whether to store any memories")
  memories: List[MemoryElement] = Field(default_factory=list, description="Atomic user memories to store")

In [160]:
memory_extractor2 = memory_llm.with_structured_output(MemoryDecision2)

In [161]:
MEMORY_PROMPT = """You are responsible for updating and maintaining accurate user memory.

CURRENT USER DETAILS (existing memories):
{user_details}

TASK:
- Review the user's latest message.
- Extract user-specific info worth storing long-term (identity, stable preferences, ongoing projects/goals).
- For each extracted item, set is_new=true ONLY if it adds NEW information compared to CURRENT USER DETAILS.
- If it is basically the same meaning as something already present, set is_new=false.
- Keep each memory as a short atomic sentence.
- No speculation; only facts stated by the user.
- If there is nothing memory-worthy, return an empty list.
"""

In [162]:
def chat_that_creates_memory_node(state: MessagesState, config: RunnableConfig,  store: BaseStore):
  user_id = config["configurable"]["user_id"]

  namespace = ("user", user_id, "details")

  # load existing memories
  memories = store.search(namespace)
  existing_memories = [memory.value.get("data", "") for memory in memories if memory.value.get("data")]
  user_details = "\n".join(f"- {memory}" for memory in existing_memories) if existing_memories else "(empty)"

  # latest user message
  latest_msg = state["messages"][-1]

  # extract memories and mark new or duplicate
  decision:MemoryDecision2 = memory_extractor2.invoke([
      SystemMessage(content=MEMORY_PROMPT.format(user_details=user_details)),
      {"role": "user", "content": f"USER MESSAGE:\n{latest_msg}"}
  ])

  # only store new memories
  if decision.should_write:
    for memory in decision.memories:
      if memory.is_new:
        print("> new memory detected, storing now...")
        store3.put(namespace, str(uuid.uuid4()), {"data": memory.text})
      else:
        print("> not a new memory, skipping...")

  return {"messages": [{"role": "assistant", "content": "Noted."}]}

In [163]:
graph3 = StateGraph(MessagesState)

graph3.add_node("chat_node", chat_that_creates_memory_node)

graph3.add_edge(START, "chat_node")
graph3.add_edge("chat_node", END)

workflow3 = graph3.compile(store=store3)

In [164]:
config3 = {"configurable": {"user_id": "u1"}}

res = workflow3.invoke({"messages": [{"role": "user", "content": "Hi, My name is James Bond."}]}, config3)
print("Assistant:", res["messages"][-1].content)

> new memory detected, storing now...
Assistant: Noted.


In [165]:
res2 = workflow3.invoke({"messages": [{"role": "user", "content": "My code name is 007."}]}, config3)
print("Assistant:", res2["messages"][-1].content)

> new memory detected, storing now...
Assistant: Noted.


In [166]:
# passing same information
res3 = workflow3.invoke({"messages": [{"role": "user", "content": "My code name is 007."}]}, config3)
print("Assistant:", res3["messages"][-1].content)

Assistant: Noted.


In [167]:
res4 = workflow3.invoke({"messages": [{"role": "user", "content": "I like Aston Martin cars."}]}, config3)
print("Assistant:", res4["messages"][-1].content)

> new memory detected, storing now...
Assistant: Noted.


In [168]:
memories3 = store3.search(("user", "u1", "details"))
print("-"*36,"MEMORIES","-"*36)
for memory in memories3:
  print(memory.value)

------------------------------------ MEMORIES ------------------------------------
{'data': "User's name is James Bond."}
{'data': "User's code name is 007."}
{'data': 'User likes Aston Martin cars.'}


### Merged workflow

In [169]:
store4 = InMemoryStore()

In [175]:
def memory_creator_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
  user_id = config["configurable"]["user_id"]
  ns = ("user", user_id, "details")

  # existing memory
  memories = store4.search(ns)
  user_details = "\n".join(memory.value["data"] for memory in memories) if memories else "(empty)"

  # last user message
  last_msg = state["messages"][-1].content

  decision: MemoryDecision2 = memory_extractor2.invoke([
      SystemMessage(content=MEMORY_PROMPT.format(user_details=user_details)),
      {"role": "user", "content": last_msg}
  ])

  if decision.should_write:
    for memory in decision.memories:
      if memory.is_new:
        store4.put(ns, str(uuid.uuid4()), {"data": memory.text})

  return {} # no message change


In [171]:
def chat_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
  user_id = config["configurable"]["user_id"]
  ns = ("user", user_id, "details")

  memories = store.search(ns)

  # convert memory items into string blob
  if memories:
    user_details = "\n".join(f"- {memory.value.get("data", "")}" for memory in memories)
  else:
    user_details = ""

  system_prompt = SYSTEM_PROMPT.format(user_details=user_details)

  system_msg = SystemMessage(content=system_prompt)

  response = model.invoke([system_msg] + state["messages"])
  return {"messages": [response]}


In [176]:
# graph
graph4 = StateGraph(MessagesState)

graph4.add_node("memory_creator_node", memory_creator_node)
graph4.add_node("chat_node", chat_node)

graph4.add_edge(START, "memory_creator_node")
graph4.add_edge("memory_creator_node", "chat_node")
graph4.add_edge("chat_node", END)

workflow4 = graph4.compile(store=store4)

In [173]:
# test
config4 = {"configurable": {"user_id": "u1"}}

In [177]:
resp1 = workflow4.invoke({"messages": [{"role": "user", "content": "Hi, My name is James Bond."}]}, config=config4)
print(resp1["messages"][-1].content)

Hi, James! It’s great to meet you. How can I assist you today?


In [179]:
resp2 = workflow4.invoke({"messages": [{"role": "user", "content": "I'm MI6 agent."}]}, config=config4)
print(resp2["messages"][-1].content)

Hello, James! It’s great to connect with you. As an MI6 agent, I’m sure you have some fascinating experiences and insights. How can I assist you today? 

If you’d like, we could dive into topics like intelligence strategies, your favorite gadgets, or even some travel tips for your missions!


In [180]:
memories4 = store4.search(("user", "u1", "details"))
print("-"*36,"MEMORIES","-"*36)
for memory in memories4:
  print(memory.value)

------------------------------------ MEMORIES ------------------------------------
{'data': "User's name is James Bond."}
{'data': 'User is an MI6 agent.'}


In [181]:
resp3 = workflow4.invoke({"messages": [{"role": "user", "content": "Tell me about MI6."}]}, config=config4)
print(resp3["messages"][-1].content)

Sure, James! MI6, officially known as the Secret Intelligence Service (SIS), is the United Kingdom's foreign intelligence service. It’s responsible for providing the government with foreign intelligence to help inform decisions on national security, foreign policy, and defense. Here are some key points about MI6:

1. **History**: MI6 was established in 1909 to oversee secret intelligence operations in the UK and overseas. It originally focused on countering Russian intelligence activities.

2. **Responsibilities**: MI6 is tasked with gathering intelligence from outside the UK to protect national interests. This includes counter-terrorism, cyber threat intelligence, and monitoring international illicit trade and espionage.

3. **Structure**: The agency is headed by a Chief and has several divisions focused on different areas of intelligence gathering and operations.

4. **Collaboration**: MI6 often collaborates with other intelligence agencies, both within the UK (like MI5 and GCHQ) and

In [182]:
memories4 = store4.search(("user", "u1", "details"))
print("-"*36,"MEMORIES","-"*36)
for memory in memories4:
  print(memory.value)

------------------------------------ MEMORIES ------------------------------------
{'data': "User's name is James Bond."}
{'data': 'User is an MI6 agent.'}
